In [ ]:
# Libraries
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from sklearn.model_selection import train_test_split
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
from sklearn.ensemble import StackingRegressor, RandomForestRegressor
from sklearn.neural_network import MLPRegressor
from sklearn.neighbors import KNeighborsRegressor
from sklearn.linear_model import LinearRegression
from xgboost import XGBRegressor
from sklearn.preprocessing import MinMaxScaler
from sklearn.preprocessing import StandardScaler


In [ ]:
# Load the dataset
df = pd.read_excel("project_dataset.xlsx")

In [ ]:
# Separate features and target
X = df.drop(columns=["Country", "BiodiversityIndex"])
y = df["BiodiversityIndex"].clip(0, 1)
countries = df["Country"]

In [ ]:
# Split the data into training and testing sets
X_train, X_test, y_train, y_test, countries_train, countries_test = train_test_split(
    X, y, countries, test_size=0.2, random_state=42
)

In [ ]:
#scaler = MinMaxScaler()
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

In [ ]:
# Define base models using exact parameters from previous codes
rf_model = RandomForestRegressor(n_estimators=100, max_depth=10, random_state=42)
mlp_model = MLPRegressor(hidden_layer_sizes=(64, 32), activation='relu', max_iter=500, random_state=42)
knn_model = KNeighborsRegressor(n_neighbors=5)
xgb_model = XGBRegressor(objective='reg:squarederror', n_estimators=100, max_depth=3, learning_rate=0.2, random_state=42)

In [ ]:
# Define stacking model
base_models = [
    ('rf', rf_model),
    ('mlp', mlp_model),
    ('knn', knn_model),
    ('xgb', xgb_model)
]

stacking_model = StackingRegressor(
    estimators=base_models,
    final_estimator=LinearRegression(),
    passthrough=False,
    n_jobs=-1,
    cv=5
)

In [ ]:
# Fit and predict using scaled data
stacking_model.fit(X_train_scaled, y_train)
y_pred = stacking_model.predict(X_test_scaled)
y_pred = np.clip(y_pred, 0, 1)

In [ ]:
# Evaluation
mae = mean_absolute_error(y_test, y_pred)
rmse = np.sqrt(mean_squared_error(y_test, y_pred))
r2 = r2_score(y_test, y_pred)

print("MAE:", round(mae, 4))
print("RMSE:", round(rmse, 4))
print("R² Score:", round(r2, 4))

In [ ]:
results_df = pd.DataFrame({
    "Country": countries_test.values,
    "Actual": y_test.values,
    "Predicted": y_pred,
    "Absolute Error": np.abs(y_test.values - y_pred)
})

In [ ]:
# Actual vs Predicted (Horizontal Bar Chart)
x = np.arange(len(results_df))
height = 0.35

plt.figure(figsize=(10, 16))
plt.barh(x - height/2, results_df["Actual"], height, label='Actual', color='seagreen', alpha=0.8)
plt.barh(x + height/2, results_df["Predicted"], height, label='Predicted', color='skyblue', alpha=0.8)

for i, v in enumerate(results_df["Actual"]):
    plt.text(v + 0.01, i - height/2, f"{v:.4f}", va='center', fontsize=8)
for i, v in enumerate(results_df["Predicted"]):
    plt.text(v + 0.01, i + height/2, f"{v:.4f}", va='center', fontsize=8)

plt.yticks(x, results_df["Country"], fontsize=8)
plt.xlabel("Biodiversity Index")
plt.title("Actual vs Predicted Biodiversity Index (Stacking + Linear Regression)")
plt.grid(axis='x', linestyle='--', alpha=0.5)
plt.legend(loc='lower right')
plt.tight_layout()
plt.show()

In [ ]:
# Absolute Error per Country
plt.figure(figsize=(18, 7))
plt.bar(results_df["Country"], results_df["Absolute Error"], color="salmon")

for i, v in enumerate(results_df["Absolute Error"]):
    plt.text(i, v + 0.005, f"{v:.4f}", color='black', ha='center', fontsize=8)

plt.xticks(rotation=90)
plt.ylabel("Absolute Error")
plt.xlabel("Country")
plt.title("Prediction Error (Absolute Error, Test Countries) - Stacking + Linear Regression")
plt.tight_layout()
plt.show()